# System Resource Monitor

This notebook samples Windows system resources on a fixed interval for a fixed total
duration and writes the results to two Excel workbooks: a dashboard that always holds
the latest snapshot and a log that grows for the full run. Every row on every sheet of
both files carries the sample timestamp.

## Config

This is the only cell you adjust. It holds the sample interval, the total run duration,
and the two output file paths. Every later section reads from these values, so changing
them here changes the whole run. The defaults are a 5 minute interval over 24 hours,
which is 288 samples.

In [1]:
from datetime import timedelta
from pathlib import Path

# How long to wait between samples.
SAMPLE_INTERVAL = timedelta(minutes=5)

# How long the whole run lasts. After this elapses the loop stops.
TOTAL_DURATION = timedelta(hours=24)

# Output file paths. Do not change these names.
DASHBOARD_PATH = Path("outputs") / "dashboard" / "dashboard.xlsx"
LOG_PATH = Path("outputs") / "log" / "log.xlsx"

# Make sure the output folders exist.
DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Derived: total number of samples this run will take.
SAMPLE_COUNT = int(TOTAL_DURATION / SAMPLE_INTERVAL)

print("Sample interval :", SAMPLE_INTERVAL)
print("Total duration  :", TOTAL_DURATION)
print("Planned samples :", SAMPLE_COUNT)
print("Dashboard file  :", DASHBOARD_PATH)
print("Log file        :", LOG_PATH)

Sample interval : 0:05:00
Total duration  : 1 day, 0:00:00
Planned samples : 288
Dashboard file  : outputs\dashboard\dashboard.xlsx
Log file        : outputs\log\log.xlsx


## Dependencies setup

This section makes the notebook run on a fresh machine that has nothing installed beyond
Python itself. It checks for each package the notebook needs and installs or upgrades only
what is missing or too old, so running it twice does no harm. The packages are `psutil`
for reading system resources and `openpyxl` for writing the Excel files.

A minimum version is enforced for each package, not just its presence. This matters
because some bundled Python setups (Anaconda for example) ship an old `psutil` that fails
on Python 3.12 when reading disks. The cell upgrades those automatically.

Two notes:

- Python 3 must already be installed, because the notebook needs Python to run at all. If
  you are on a machine with no Python, install Python 3 from python.org first, then open
  this notebook.
- If this cell reports that it upgraded or installed anything, restart the kernel and run
  the notebook again from the top, so the new version is the one actually loaded.


In [2]:
import importlib
import importlib.metadata as metadata
import subprocess
import sys

# Import name mapped to its pip name and the minimum version the notebook needs.
REQUIRED_PACKAGES = {
    "psutil": {"pip": "psutil", "min": "5.9.6"},
    "openpyxl": {"pip": "openpyxl", "min": "3.0.0"},
}


def _version_tuple(text):
    """Turn a version string like '5.9.6' into a comparable tuple of integers."""
    parts = []
    for chunk in text.split("."):
        digits = ""
        for char in chunk:
            if char.isdigit():
                digits += char
            else:
                break
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def _pip_install(spec):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])


def ensure_packages(packages):
    """Install or upgrade each package so it is present and meets the minimum version."""
    print("Python version:", sys.version.split()[0])
    print("-" * 50)

    changed = False
    for import_name, info in packages.items():
        pip_name = info["pip"]
        minimum = info["min"]
        try:
            current = metadata.version(pip_name)
        except metadata.PackageNotFoundError:
            current = None

        if current is None:
            print("NOT INSTALLED  :", pip_name, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        elif _version_tuple(current) < _version_tuple(minimum):
            print("OUTDATED       :", pip_name, current, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        else:
            print("OK             :", pip_name, current)

    print("-" * 50)
    if changed:
        print("Packages were installed or upgraded.")
        print("Restart the kernel and run the notebook again from the top.")
    else:
        print("All required packages are present and up to date.")


ensure_packages(REQUIRED_PACKAGES)


Python version: 3.12.7
--------------------------------------------------
OK             : psutil 7.2.2
OK             : openpyxl 3.1.5
--------------------------------------------------
All required packages are present and up to date.


## CPU usage

This section reads CPU load. It returns the overall busy percentage and the busy
percentage of each logical core. The reading uses a one second measurement window
(`interval=1`) so the value reflects real activity rather than the zero you get from an
instant call. The overall figure is the mean of the per-core figures, so the two always
agree. The result is a flat dictionary of named fields plus the sample timestamp, which
is the shape every system-metrics collector in this notebook returns.

In [3]:
import psutil
from datetime import datetime


def collect_cpu(timestamp):
    """Return overall and per-core CPU usage for one sample, in plain labels."""
    per_core = psutil.cpu_percent(interval=1, percpu=True)
    overall = round(sum(per_core) / len(per_core), 1) if per_core else 0.0

    row = {
        "Time": timestamp,
        "Overall CPU Usage (%)": overall,
        "Number of CPU Cores": len(per_core),
    }
    for index, value in enumerate(per_core):
        # Cores numbered from 1 so the labels read naturally.
        row["Core {} Usage (%)".format(index + 1)] = value
    return row


# Quick check.
collect_cpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 10, 14, 51, 53, 884289),
 'Overall CPU Usage (%)': 32.4,
 'Number of CPU Cores': 8,
 'Core 1 Usage (%)': 52.2,
 'Core 2 Usage (%)': 18.8,
 'Core 3 Usage (%)': 70.3,
 'Core 4 Usage (%)': 33.3,
 'Core 5 Usage (%)': 28.1,
 'Core 6 Usage (%)': 14.1,
 'Core 7 Usage (%)': 21.9,
 'Core 8 Usage (%)': 20.6}

## CPU temperature

This section reads the CPU thermal sensor. Windows does not expose CPU temperature
through psutil, so the reading is pulled from the WMI thermal zone using a short
PowerShell call. WMI reports the value in tenths of a Kelvin, which this code converts to
degrees Celsius. Many laptops block this sensor or return access denied. When that
happens the field is recorded as `Unavailable` for that sample rather than letting the
error stop the run, so the timestamp and the rest of the sample are still captured.

In [4]:
import subprocess
from datetime import datetime


def _run_powershell(command, timeout=15):
    """Run a PowerShell command and return its text output, or None on failure."""
    try:
        result = subprocess.run(
            ["powershell", "-NoProfile", "-NonInteractive", "-Command", command],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    return None


def collect_temperature(timestamp):
    """Return the CPU temperature in Celsius, or Unavailable if the sensor is blocked."""
    command = (
        "Get-CimInstance -Namespace root/wmi "
        "-ClassName MSAcpi_ThermalZoneTemperature -ErrorAction Stop "
        "| Select-Object -ExpandProperty CurrentTemperature"
    )
    output = _run_powershell(command)

    temperature = "Unavailable"
    if output:
        readings = []
        for line in output.splitlines():
            line = line.strip()
            if line.isdigit():
                # WMI reports tenths of a Kelvin. Convert to Celsius.
                readings.append(int(line) / 10.0 - 273.15)
        if readings:
            temperature = round(sum(readings) / len(readings), 1)

    return {
        "Time": timestamp,
        "CPU Temperature (°C)": temperature,
    }


# Quick check.
collect_temperature(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 51, 54, 903493),
 'CPU Temperature (°C)': 'Unavailable'}

## RAM

This section reads system memory. It reports the total memory fitted, how much is in use,
how much is still available, and the used percentage. The raw figures from the system are
in bytes, which are hard to read, so they are converted to gigabytes (GB) and rounded.
Available memory is the amount that can be handed to programs without swapping, which is
the number most people care about when asking how much memory is free.

In [5]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def to_gb(value):
    """Convert a byte count to gigabytes, rounded to two decimals."""
    return round(value / BYTES_PER_GB, 2)


def collect_ram(timestamp):
    """Return total, used, available memory in GB and the used percentage."""
    memory = psutil.virtual_memory()
    return {
        "Time": timestamp,
        "Total Memory (GB)": to_gb(memory.total),
        "Used Memory (GB)": to_gb(memory.used),
        "Available Memory (GB)": to_gb(memory.available),
        "Memory Usage (%)": memory.percent,
    }


# Quick check.
collect_ram(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 334593),
 'Total Memory (GB)': 7.71,
 'Used Memory (GB)': 6.62,
 'Available Memory (GB)': 1.09,
 'Memory Usage (%)': 85.9}

## Disk

This section reads storage usage for every drive on the machine. Unlike the single
system-metrics row, disk returns one row per drive, so it gets its own sheet in both
Excel files. Each row carries the sample timestamp, the drive letter, total size, used
space, free space (all in gigabytes), and the used percentage. Some entries such as an
empty CD drive or card reader raise an error when read. Those are skipped so they do not
stop the sample, and the drives that do respond are still recorded.

In [6]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def _gb(value):
    return round(value / BYTES_PER_GB, 2)


def collect_disk(timestamp):
    """Return one row per drive with total, used, free space and used percentage."""
    rows = []
    for partition in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(partition.mountpoint)
        except Exception:
            # Empty CD drive, card reader, or a drive psutil cannot read on Windows
            # (this can raise PermissionError, OSError, or SystemError). Skip it so
            # one bad drive does not stop the sample.
            continue
        rows.append({
            "Time": timestamp,
            "Drive": partition.device,
            "Total Space (GB)": _gb(usage.total),
            "Used Space (GB)": _gb(usage.used),
            "Free Space (GB)": _gb(usage.free),
            "Disk Usage (%)": usage.percent,
        })
    return rows


# Quick check.
collect_disk(datetime.now())


[{'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 357006),
  'Drive': 'C:\\',
  'Total Space (GB)': 476.23,
  'Used Space (GB)': 352.83,
  'Free Space (GB)': 123.4,
  'Disk Usage (%)': 74.1},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 357006),
  'Drive': 'D:\\',
  'Total Space (GB)': 237.86,
  'Used Space (GB)': 72.83,
  'Free Space (GB)': 165.03,
  'Disk Usage (%)': 30.6}]

## Battery

This section reads the battery. It reports the charge percentage, whether the laptop is
plugged in, and the estimated time left on battery. The estimate is given by the system in
seconds, which is converted here to a plain reading like `2 hours 15 minutes`. When the
machine is plugged in or the system cannot estimate the time, that is stated in words
rather than shown as a confusing number. On a desktop with no battery the fields read
`No battery` so the row still records cleanly with its timestamp.

In [7]:
import psutil
from datetime import datetime


def _format_time_left(seconds):
    """Turn a seconds-remaining value into a plain reading."""
    if seconds == psutil.POWER_TIME_UNLIMITED:
        return "Plugged in"
    if seconds == psutil.POWER_TIME_UNKNOWN or seconds is None or seconds < 0:
        return "Unknown"
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return "{} hours {} minutes".format(hours, minutes)


def collect_battery(timestamp):
    """Return battery charge, plugged-in state and estimated time remaining."""
    battery = psutil.sensors_battery()
    if battery is None:
        return {
            "Time": timestamp,
            "Battery Charge (%)": "No battery",
            "Power Plugged In": "No battery",
            "Estimated Time Remaining": "No battery",
        }
    return {
        "Time": timestamp,
        "Battery Charge (%)": round(battery.percent, 1),
        "Power Plugged In": "Yes" if battery.power_plugged else "No",
        "Estimated Time Remaining": (
            "Plugged in" if battery.power_plugged
            else _format_time_left(battery.secsleft)
        ),
    }


# Quick check.
collect_battery(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 374297),
 'Battery Charge (%)': 33,
 'Power Plugged In': 'No',
 'Estimated Time Remaining': '0 hours 34 minutes'}

## Network

This section reads network traffic for every network interface, so it returns one row per
interface and gets its own sheet. The system only reports running totals since the machine
booted, so this code remembers the previous sample's totals and reports the difference,
which is the data actually sent and received during the last interval. Both the per-sample
amounts and the running totals are shown, converted from bytes to megabytes (MB) for
readability. On the very first sample there is no previous total to compare against, so the
per-sample amounts read as 0.0 by design.

In [8]:
import psutil
from datetime import datetime

BYTES_PER_MB = 1024 ** 2

# Remembers the previous sample's byte counters per interface, so we can report
# the difference rather than the running total. Persists between samples.
_previous_net_counters = {}


def _mb(value):
    return round(value / BYTES_PER_MB, 3)


def collect_network(timestamp):
    """Return per-interface data sent and received since the last sample, plus totals."""
    rows = []
    counters = psutil.net_io_counters(pernic=True)
    for interface, stats in counters.items():
        previous = _previous_net_counters.get(interface)
        if previous is None:
            sent_delta = 0
            recv_delta = 0
        else:
            sent_delta = max(0, stats.bytes_sent - previous.bytes_sent)
            recv_delta = max(0, stats.bytes_recv - previous.bytes_recv)
        rows.append({
            "Time": timestamp,
            "Interface": interface,
            "Data Sent Since Last Sample (MB)": _mb(sent_delta),
            "Data Received Since Last Sample (MB)": _mb(recv_delta),
            "Total Data Sent (MB)": _mb(stats.bytes_sent),
            "Total Data Received (MB)": _mb(stats.bytes_recv),
        })
        _previous_net_counters[interface] = stats
    return rows


# Quick check. Run it twice to see real per-sample amounts on the second call.
collect_network(datetime.now())

[{'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 389316),
  'Interface': 'Ethernet',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 389316),
  'Interface': 'Ethernet 2',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 389316),
  'Interface': 'Local Area Connection* 1',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 389316),
  'Interface': 'Local Area Connection* 2',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Rece

## GPU

This section reads graphics card usage. This machine has Intel Iris Xe integrated
graphics and no NVIDIA tooling, so the usual `nvidia-smi` route does not apply. Instead it
reads the Windows GPU performance counters through a short PowerShell call, which works for
any GPU vendor including Intel. GPU usage is the combined activity across the graphics
engines, capped at 100 percent, and GPU memory used is the memory currently held by
processes, shown in megabytes. If the counters cannot be read the fields record
`Unavailable` for that sample so the run continues. This reuses the PowerShell helper
defined in the CPU temperature section, so run the notebook from the top.


In [9]:
from datetime import datetime

# PowerShell that sums GPU engine utilization and GPU process memory across all
# instances. Each value is printed on its own labelled line, or NA if unreadable.
_GPU_COMMAND = r"""
try {
    $u = ((Get-Counter '\GPU Engine(*)\Utilization Percentage' -ErrorAction Stop).CounterSamples |
          Measure-Object -Property CookedValue -Sum).Sum
} catch { $u = 'NA' }
try {
    $m = ((Get-Counter '\GPU Process Memory(*)\Local Usage' -ErrorAction Stop).CounterSamples |
          Measure-Object -Property CookedValue -Sum).Sum
} catch { $m = 'NA' }
Write-Output ("UTIL=" + $u)
Write-Output ("MEM=" + $m)
"""


def collect_gpu(timestamp):
    """Return GPU usage percent and GPU memory used in MB, or Unavailable."""
    output = _run_powershell(_GPU_COMMAND, timeout=20)

    usage = "Unavailable"
    memory_mb = "Unavailable"
    if output:
        for line in output.splitlines():
            line = line.strip()
            if line.startswith("UTIL="):
                value = line[5:].strip().replace(",", ".")
                try:
                    usage = min(100.0, round(float(value), 1))
                except ValueError:
                    pass
            elif line.startswith("MEM="):
                value = line[4:].strip().replace(",", ".")
                try:
                    memory_mb = round(float(value) / (1024 ** 2), 1)
                except ValueError:
                    pass

    return {
        "Time": timestamp,
        "GPU Usage (%)": usage,
        "GPU Memory Used (MB)": memory_mb,
    }


# Quick check.
collect_gpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 10, 14, 51, 55, 413218),
 'GPU Usage (%)': 1.4,
 'GPU Memory Used (MB)': 873.3}

## System uptime and boot time

This section reports when the machine last started up and how long it has been running
since. The system gives the boot time as a raw timestamp, which is converted here to a
plain date and time. Uptime is the gap between now and that boot time, shown in days,
hours and minutes so it is easy to read at a glance.

In [10]:
import psutil
from datetime import datetime


def _format_uptime(seconds):
    """Turn a number of seconds into days, hours and minutes."""
    seconds = int(seconds)
    days = seconds // 86400
    hours = (seconds % 86400) // 3600
    minutes = (seconds % 3600) // 60
    return "{} days {} hours {} minutes".format(days, hours, minutes)


def collect_uptime(timestamp):
    """Return the boot time and how long the machine has been running."""
    boot = datetime.fromtimestamp(psutil.boot_time())
    uptime_seconds = (timestamp - boot).total_seconds()
    return {
        "Time": timestamp,
        "Last Boot Time": boot.strftime("%Y-%m-%d %H:%M:%S"),
        "Uptime": _format_uptime(uptime_seconds),
    }


# Quick check.
collect_uptime(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 250491),
 'Last Boot Time': '2026-06-10 08:22:40',
 'Uptime': '0 days 6 hours 29 minutes'}

## Processes

This section lists every program currently running, so it returns one row per process and
gets its own sheet. Each row shows the process id, the program name, how much CPU it is
using, how much memory it holds in megabytes, and its status such as running or sleeping.
Some processes are protected by Windows and cannot be read; those are skipped so they do
not stop the sample. The CPU figure is measured since the previous sample, so on the very
first sample it reads 0.0 for every process, then shows real values from the second sample
onward. To keep the check below short it prints only the first five rows and the total
count.

In [11]:
import psutil
from datetime import datetime

BYTES_PER_MB = 1024 ** 2

# Number of logical cores. Used to convert per-process CPU into a 0 to 100 share of
# total CPU capacity, the way Task Manager shows it, instead of a per-core figure that
# can climb above 100 on a multi-core machine.
CPU_CORE_COUNT = psutil.cpu_count() or 1


def collect_processes(timestamp):
    """Return one row per running process with cpu, memory and status."""
    rows = []
    for proc in psutil.process_iter(["pid", "name", "cpu_percent", "memory_info", "status"]):
        try:
            info = proc.info
            memory = info.get("memory_info")
            memory_mb = round(memory.rss / BYTES_PER_MB, 1) if memory else 0.0
            name = info.get("name") or "Unknown"
            cpu_share = (info.get("cpu_percent") or 0.0) / CPU_CORE_COUNT
            rows.append({
                "Time": timestamp,
                "Process ID (PID)": info.get("pid"),
                "Process Name": name,
                "CPU Usage (%)": round(cpu_share, 1),
                "Memory Usage (MB)": memory_mb,
                "Status": info.get("status") or "Unknown",
            })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            # Process ended or is protected by Windows. Skip it.
            continue
    return rows


# Quick check: show the first five rows and the total count.
_rows = collect_processes(datetime.now())
print("Total processes:", len(_rows))
_rows[:5]


Total processes: 359


[{'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 269067),
  'Process ID (PID)': 0,
  'Process Name': 'System Idle Process',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 0.0,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 269067),
  'Process ID (PID)': 4,
  'Process Name': 'System',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 0.1,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 269067),
  'Process ID (PID)': 104,
  'Process Name': 'Unknown',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 23.3,
  'Status': 'stopped'},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 269067),
  'Process ID (PID)': 164,
  'Process Name': 'Registry',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 31.8,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 10, 14, 51, 58, 269067),
  'Process ID (PID)': 324,
  'Process Name': 'bash.exe',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 4.0,
  'Status': 'running'}]